In [1]:
import pandas as pd
import numpy as np
import random
import pickle
import sys
import os
import logging
from os.path import join
import torch
from sklearn.model_selection import KFold
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.metrics import matthews_corrcoef
import matplotlib.pyplot as plt

sys.path.append('./additional_code')
#from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + '/../data/our_data/'

/home/hanxd/Repositories/ESP/our_codes


In [2]:
anotherdata = pd.read_csv(our_data + 'ginnascores.txt', sep='\t', header=0)
result = anotherdata['complex_name'].str.split(r'_', expand=True)
result.columns = ['enzyme', 'substrate', 'cc']
result['substrate_cc'] = result['substrate'] + result['cc']

anotherdata = pd.concat([anotherdata, result], axis=1)

In [3]:
anotherdata

,complex_name,wei_score,if_right,enzyme,substrate,cc,substrate_cc
0,CYP707A1_08Y_C22,0.345,1,CYP707A1,08Y,C22,08YC22
1,CYP76B10_08Y_C22,0.387,1,CYP76B10,08Y,C22,08YC22
2,CYP72A208_08Y_C22,0.369,1,CYP72A208,08Y,C22,08YC22
3,CYP725A4_08Y_C22,0.341,1,CYP725A4,08Y,C22,08YC22
4,CYP71AY5_08Y_C22,0.395,1,CYP71AY5,08Y,C22,08YC22
...,...,...,...,...,...,...,...
75056,CYP72A610_AHT_C8,0.300,1,CYP72A610,AHT,C8,AHTC8
75057,CYP74L1_AHT_C8,0.249,1,CYP74L1,AHT,C8,AHTC8
75058,CYP79A12_AHT_C8,0.248,1,CYP79A12,AHT,C8,AHTC8
75059,CYP71D12_AHT_C8,0.229,1,CYP71D12,AHT,C8,AHTC8


In [4]:
print(anotherdata['enzyme'].nunique())
print(anotherdata['substrate'].nunique())
print(anotherdata['complex_name'].nunique())
print(anotherdata['substrate_cc'].nunique())

294
252
75061
311


In [5]:
anotherdata['rank'] = anotherdata.groupby('substrate_cc')['wei_score'].rank(method='first', ascending=False)

In [6]:
binding_1_values = anotherdata[anotherdata['if_right'] == 2]

In [7]:
binding_1_values['ranking'] = binding_1_values['rank']/444
ratios_list=binding_1_values['ranking'].median()

/tmp/ipykernel_1341304/1518515439.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  binding_1_values['ranking'] = binding_1_values['rank']/444


In [8]:
ratios_list

0.04504504504504504

In [9]:
binding_1_values

,complex_name,wei_score,if_right,enzyme,substrate,cc,substrate_cc,rank,ranking
122,3UA1_08Y_C22,0.237,2,3UA1,08Y,C22,08YC22,170.0,0.382883
275,3V8D_0GV_C18,0.411,2,3V8D,0GV,C18,0GVC18,19.0,0.042793
483,4EJH_0QA_C9,0.526,2,4EJH,0QA,C9,0QAC9,8.0,0.018018
725,5FYG_12H_C7,0.503,2,5FYG,12H,C7,12HC7,12.0,0.027027
1126,2NNJ_225_C11,0.439,2,2NNJ,225,C11,225C11,10.0,0.022523
...,...,...,...,...,...,...,...,...,...
74434,2JJO_EY5_C3,0.211,2,2JJO,EY5,C3,EY5C3,216.0,0.486486
74534,CYP72A65v2_BAM_C29,0.319,2,CYP72A65v2,BAM,C29,BAMC29,9.0,0.020270
74766,CYP706X1_AGI_C9,0.552,2,CYP706X1,AGI,C9,AGIC9,2.0,0.004505
74776,CYP82D1_AGI_C9,0.435,2,CYP82D1,AGI,C9,AGIC9,13.0,0.029279


In [19]:
shixiong = pd.read_csv(our_data + 'cache/1.txt', sep='\t', header=None)

In [20]:
shixiong

,0
0,0GVC18
1,0QAC9
2,12HC7
3,225C11
4,2ION2
...,...
152,VD3C25
153,VGJC20
154,XANC6
155,XBKC20


In [21]:
substrate_cc = anotherdata['substrate_cc'].unique()
flag2 = binding_1_values['substrate_cc'].unique()
shixiong = shixiong[0].unique()
elements_only_in_substrate_cc = np.setdiff1d(flag2, shixiong)

In [22]:
elements_only_in_substrate_cc

array(['08YC22', '4UHC15', 'BAMC15', 'BAMC18', 'BAMC23', 'BAMC8',
       'CASC11', 'CATC12', 'COUC14', 'CTNC10', 'DCTC13', 'DXJC10',
       'EPDC3', 'ESAC7', 'EY5C3', 'FACC14', 'FLIC26', 'FRIC28', 'GEAC14',
       'GENC12', 'HMEC6', 'HYOC26', 'JAIC12', 'JZ3C7', 'LOGC4', 'MASC7',
       'MBDC2', 'MITC13', 'NMEC17', 'OACC25', 'PDIC17', 'PHBC9', 'PIOC1',
       'PLOC17', 'Q4MC18', 'SABC6', 'STRC1', 'TABC17', 'TDOC15', 'TEPC3',
       'THYC6', 'TRBC15'], dtype=object)